# Synthea Explorations

This notebook explores synthetic patient data generated by [Synthea](https://github.com/synthetichealth/synthea).
Synthea outputs FHIR R4 Bundles containing Patient, Practitioner, Encounter, Condition, and other resources.

In [ ]:
import os

from dicom_py_mock_server.services.synthea_utils import SyntheaUtils

SYNTHEA_OUTPUT_FOLDER = "../../../synthea/output/fhir"
MAX_BUNDLES = 75


# Check if the output folder exists
if not os.path.exists(SYNTHEA_OUTPUT_FOLDER):
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' does not exist.")
else:
    print(f"Output folder '{SYNTHEA_OUTPUT_FOLDER}' found.")

# Load the Synthea output data, parse them into a list of FHIR Bundle dicts, and return the list

syn = SyntheaUtils(synthea_output_folder=SYNTHEA_OUTPUT_FOLDER)

In [ ]:
from ast import List

from dicom_py_mock_server.models.fhir_models import (
    FhirOrganization,
    FhirPractitioner,
    format_human_name_to_dicom,
)

hospitals: List[FhirOrganization] = syn.get_hospital_information()
org: FhirOrganization
practitioners: List[FhirPractitioner]
p: FhirPractitioner
practitioners = syn.get_practioner_information()
if not hospitals:
    print("Failed to get hospitals")
if not practitioners:
    print("Failed to get practitioners")
for org in hospitals:
    # print(f"{org.id}: {org.name} <- {org.partOf}")
    pass
for p in practitioners:
    name = format_human_name_to_dicom(p.name[0])
    ident = p.identifier[0].value
    print(f"{name} NPI: {ident}")

In [ ]:
from ast import List

from fhir.resources.R4B.diagnosticreport import DiagnosticReport
from fhir.resources.R4B.patient import Patient
from fhir.resources.R4B.procedure import Procedure

bundles = syn.scan_synthea_output(max_count=MAX_BUNDLES)

print(f"Loaded {len(bundles)} FHIR Bundles from Synthea output.")

for bundle in bundles:
    patient: Patient = syn.get_patient(bundle)
    dicom_name = syn.get_patient_name(patient)
    patient_id = syn.get_patient_id(patient)
    sex = syn.get_patient_sex(patient)
    birth_date = syn.get_date_time(patient.birthDate)
    print("\n====================================")
    print(f"Patient: DICOM Name: {dicom_name}, Gender: {sex}, ID: {patient_id}, Birth Date: {birth_date[0]}")
    # Imaging Studies
    imagings = syn.get_imaging_studies(bundle)
    for imaging in imagings:
        for s in imaging.series:
            modality = s.modality.display
            print("-- Imaging --")
            print(f"Modality: {modality}")
        enc_ref = imaging.encounter.reference
        encounters = syn.get_encounters(bundle=bundle, encounter_ref=enc_ref)
        for enc in encounters:
            print(f"Encounter On:{enc.period.start}: {enc_ref} {enc.id}")
        diags: List[DiagnosticReport] = syn.get_diagnostic_reports(bundle=bundle, encounter_ref=enc_ref)
        procedures: List[Procedure] = syn.get_procedures(bundle=bundle, encounter_ref=enc_ref)
        diag: DiagnosticReport
        proc: Procedure
        for diag in diags:
            if diag and diag.presentedForm:
                for form in diag.presentedForm:
                    match form:
                        case Attachment as a:
                            print(f"Diagnostic Report: {a.data.decode('utf-8')}")
        for proc in procedures:
            code = proc.code.text
            reason = ""
            if proc.reasonReference:
                reason = proc.reasonReference[0].display
            location = proc.location.display
            print(f"Procedure: {code}, {reason} at {location}")